# 07 — Incremental Cross-Model Generalization & Cross-Family Judge Verification

**Strictly Non-Interfering Extension**: Writes exclusively to `$DRIVE_ROOT/runs/cross_model_generalization/` and new filenames in `$DRIVE_ROOT/paper_artifacts/` (`tab_cross_model_*.tex`, `fig_cross_model_*.pdf`, `fig_cross_family_judge_agreement.pdf`). Does **NOT** overwrite or modify any output from Notebooks `00`–`06`.

**Model Families Covered**:
- **OpenAI Next-Gen & Cross-Gen**: `gpt-5.6-luna`, `gpt-5.6-terra`, `gpt-4o` (plus read-only reference to `gpt-4.1-mini` / `gpt-4.1` from Notebooks `03`/`04`)
- **AutoDL Open-Weight / Chinese Models** (`https://www.autodl.art/api/v1`, key from Drive `secrets/AUTODL_KEY.txt`): `DeepSeek-V4.1-Flash`, `GLM-5.3-flash`, `Qwen3.5-397B-A17B`

**Key Contributions**:
1. Paired `Direct Chat (B0)` vs `DIAL-Skill` generalization on **When2Speak** (strict `val_calib <= 200` per-model calibration, Invariant #8).
2. Paired `Direct Chat (B0)` vs `DIAL-Skill` generalization on **PageMantle** across OpenAI & AutoDL backbones.
3. **Cross-Family Blinded Judge Agreement** (`gpt-4.1` vs AutoDL non-OpenAI judge) eliminating self-preference bias.

In [ ]:
# === CELL 0: Mount Google Drive & Configure Persistent Workspace ===
import os
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/when2speak_research")
else:
    DRIVE_ROOT = Path("./local_drive_mirror").resolve()

for sub in [
    "secrets",
    "config_cache",
    "datasets/pagemantle",
    "datasets/when2speak",
    "datasets/pagemantle_ext",
    "caches",
    "runs/cross_model_generalization",
    "paper_artifacts/tables",
    "paper_artifacts/figures",
]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

os.environ["DIAL_SKILL_DRIVE_ROOT"] = str(DRIVE_ROOT)
print("Persistent DRIVE_ROOT:", DRIVE_ROOT)


In [ ]:
# === CELL 1: Sync GitHub Repo (git clone / git pull) & Hot-Reload ===
REPO_URL = "https://github.com/BoldHu/multiagent-speak.git"
REPO_DIR = Path("/content/when2speak") if IN_COLAB else Path(".").resolve()

import importlib, sys, types

if IN_COLAB:
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull --ff-only
    %cd {REPO_DIR}
    try:
        import typesafe_sdk, duckdb, typer, tenacity  # noqa: F401
        !pip install -q -e . --no-deps
    except ImportError:
        !pip install -q -e .

_src_dir = str((REPO_DIR / "src").resolve())
if _src_dir not in sys.path:
    sys.path.insert(0, _src_dir)

if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp_shim = types.ModuleType("imp")
        _imp_shim.reload = importlib.reload
        sys.modules["imp"] = _imp_shim

try:
    _ip = get_ipython()
    if _ip is not None:
        _ip.run_line_magic("load_ext", "autoreload")
        _ip.run_line_magic("autoreload", "2")
except Exception as _exc:
    print(f"Skipping autoreload ({_exc}); restart runtime after git pull if needed.")


In [ ]:
# === CELL 2: Bootstrap Multi-Provider Secrets & Configs (Never Prints Raw Keys) ===
import json
from dial_skill.config import load_dial_skill_config, load_experiments_config, load_models_config
from dial_skill.llm.preflight import load_resolved_models, resolve_model_for_role
from dial_skill.utils.secrets import describe_secret_sources, load_autodl_api_key

print(
    "Secret lookup diagnostics:",
    json.dumps(describe_secret_sources(drive_root=DRIVE_ROOT, include_autodl=True), indent=2),
)
MODELS_CFG = load_models_config()
DIAL_CFG = load_dial_skill_config()
EXP_CFG = load_experiments_config()
RESOLVED_MODELS = load_resolved_models(drive_root=DRIVE_ROOT)
AUTODL_BASE_URL = "https://www.autodl.art/api/v1"
AUTODL_KEY = load_autodl_api_key(drive_root=DRIVE_ROOT, required=False)
print("AutoDL endpoint:", AUTODL_BASE_URL, "| Key loaded:", bool(AUTODL_KEY))


## Step 1: Configure Incremental Model Matrix & Live Preflight Probe

Probes every candidate model with 1 tiny JSON call before running benchmarks so any unsupported/offline endpoint is cleanly skipped without crashing.

In [ ]:
import time
from dial_skill.llm.jev_client import TriageClient
from dial_skill.llm.openai_client import OpenAIClient
from dial_skill.utils.cache import SQLiteAPICache
from dial_skill.utils.checkpoint import save_run_metadata

# --- Configurable Scope (safe defaults: ~$4-$9 total, 100% Jev cache reuse!) ---
W2S_SPLIT = "pilot"       # "pilot" (400 turns, fast & $0 Jev reuse) | "main" (2,000 turns)
PM_SCOPE = "official_24"  # "official_24" (pm_official_8 + 10 action + 6 traps) | "full_40" (all 40 held-out)

# Candidate incremental models (existing gpt-4.1-mini & gpt-4.1 are loaded read-only from runs/03 & 04!)
CANDIDATE_MODELS = [
    {"id": "gpt-4o", "provider": "openai", "family": "OpenAI-4o", "run_w2s": True, "run_pm": True},
    {"id": "gpt-5.6-luna", "provider": "openai", "family": "OpenAI-5.6", "run_w2s": True, "run_pm": True},
    {"id": "gpt-5.6-terra", "provider": "openai", "family": "OpenAI-5.6", "run_w2s": True, "run_pm": True},
    {"id": "DeepSeek-V4.1-Flash", "provider": "autodl", "family": "DeepSeek", "run_w2s": True, "run_pm": True},
    {"id": "GLM-5.3-flash", "provider": "autodl", "family": "GLM", "run_w2s": True, "run_pm": True},
    {"id": "Qwen3.5-397B-A17B", "provider": "autodl", "family": "Qwen", "run_w2s": True, "run_pm": True},
]

RUN_DIR = DRIVE_ROOT / "runs" / "cross_model_generalization" / f"{W2S_SPLIT}_{PM_SCOPE}"
save_run_metadata(RUN_DIR, {"w2s_split": W2S_SPLIT, "pm_scope": PM_SCOPE, "candidates": CANDIDATE_MODELS})

cache = SQLiteAPICache(DRIVE_ROOT / "caches" / "api_cache.sqlite")
oai_client = OpenAIClient(models_cfg=MODELS_CFG, cache=cache, provider_name="openai")
autodl_client = (
    OpenAIClient(
        api_key=AUTODL_KEY,
        models_cfg=MODELS_CFG,
        cache=cache,
        base_url=AUTODL_BASE_URL,
        provider_name="autodl",
    )
    if AUTODL_KEY
    else None
)

luna_ref_model = resolve_model_for_role("luna", MODELS_CFG, RESOLVED_MODELS)
terra_judge_model = resolve_model_for_role("judge", MODELS_CFG, RESOLVED_MODELS)
triage_backend = RESOLVED_MODELS.triage_backend if RESOLVED_MODELS else MODELS_CFG.roles.triage_backend
triage_client = TriageClient(
    models_cfg=MODELS_CFG, cache=cache, openai_client=oai_client, luna_model=luna_ref_model, backend=triage_backend
)

def _client_for(provider: str) -> OpenAIClient:
    if provider == "autodl":
        if autodl_client is None:
            raise RuntimeError("AutoDL client unavailable (missing AUTODL_KEY.txt).")
        return autodl_client
    return oai_client

# Live 1-call JSON probe per model (never aborts whole suite if one model is unavailable)
HEALTHY_MODELS = []
probe_report = []
for spec in CANDIDATE_MODELS:
    mid = spec["id"]
    prov = spec["provider"]
    t0 = time.perf_counter()
    try:
        cli = _client_for(prov)
        res = await cli.complete_json(
            model=mid,
            messages=[
                {"role": "system", "content": "Return strict JSON only: {\"ok\": true}."},
                {"role": "user", "content": "ping"},
            ],
            json_schema={
                "type": "object",
                "additionalProperties": False,
                "required": ["ok"],
                "properties": {"ok": {"type": "boolean"}},
            },
            schema_name="health_probe",
            temperature=0.0,
            seed=42,
            use_cache=False,
        )
        dt = round(time.perf_counter() - t0, 2)
        ok = isinstance(res.parsed_json, dict)
        probe_report.append({"model": mid, "provider": prov, "healthy": ok, "latency_s": dt})
        if ok:
            HEALTHY_MODELS.append(spec)
        print(f"  [OK]   {prov:6s} | {mid:22s} | {dt:.2f}s | parsed={res.parsed_json}")
    except Exception as exc:
        dt = round(time.perf_counter() - t0, 2)
        probe_report.append({"model": mid, "provider": prov, "healthy": False, "error": f"{type(exc).__name__}: {str(exc)[:140]}"})
        print(f"  [SKIP] {prov:6s} | {mid:22s} | {dt:.2f}s | {type(exc).__name__}: {str(exc)[:140]}")

(RUN_DIR / "preflight_probe.json").write_text(json.dumps(probe_report, indent=2), encoding="utf-8")
print(f"\nHealthy models ready for sweep ({len(HEALTHY_MODELS)}/{len(CANDIDATE_MODELS)}): {[m['id'] for m in HEALTHY_MODELS]}")


## Step 2: Cross-Model When2Speak Generalization (`Direct Chat` vs `DIAL-Skill`, Per-Model Val Calibration)

In [ ]:
from dial_skill.baselines.direct_chat import DirectChatBaseline
from dial_skill.benchmarks.when2speak import assert_no_calibration_leakage, load_when2speak_split
from dial_skill.core.pipeline import DialSkillAgent
from dial_skill.core.skill_store import SkillStore
from dial_skill.eval.metrics import compute_when2speak_metrics, mcnemar_test
from dial_skill.eval.runners import calibrate_when2speak_thresholds, run_when2speak_eval

val_turns = load_when2speak_split("val_calib", local_dir=DRIVE_ROOT / "datasets" / "when2speak", seed=EXP_CFG["seed"])
eval_turns = load_when2speak_split(W2S_SPLIT, local_dir=DRIVE_ROOT / "datasets" / "when2speak", seed=EXP_CFG["seed"])
assert_no_calibration_leakage(val_turns, eval_turns, max_calib_turns=200)
y_true_w2s = [t.label_speak for t in eval_turns]

w2s_cross_rows = []

# Read-only import of already-completed GPT-4.1-mini & GPT-4.1 runs from Notebook 03 (zero re-billing!)
for ref_dir in [
    DRIVE_ROOT / "runs" / "main_when2speak" / W2S_SPLIT,
    DRIVE_ROOT / "runs" / "main_when2speak" / "main",
]:
    ref_sum = ref_dir / "metrics_summary.json"
    if ref_sum.is_file():
        d = json.loads(ref_sum.read_text(encoding="utf-8"))
        ms = d.get("metrics", {})
        if "B0_direct_luna" in ms and "DIAL_Skill_luna" in ms:
            w2s_cross_rows.append({
                "family": "OpenAI-4.1",
                "model": luna_ref_model,
                "source_split": d.get("mode", ref_dir.name),
                "direct_macro_f1": ms["B0_direct_luna"]["macro_f1"],
                "direct_fir": ms["B0_direct_luna"]["fir"],
                "direct_mir": ms["B0_direct_luna"]["mir"],
                "dial_macro_f1": ms["DIAL_Skill_luna"]["macro_f1"],
                "dial_speak_f1": ms["DIAL_Skill_luna"]["speak_f1"],
                "dial_fir": ms["DIAL_Skill_luna"]["fir"],
                "dial_mir": ms["DIAL_Skill_luna"]["mir"],
                "dial_auroc": ms["DIAL_Skill_luna"]["auroc"],
                "mcnemar_p": (d.get("mcnemar_dial_vs_direct") or {}).get("p_value"),
            })
        if "DIAL_Skill_terra" in ms:
            terra_dir_m = ms.get("B0_direct_terra")
            terra_mc_p = (d.get("mcnemar_terra_vs_direct") or {}).get("p_value")
            if terra_dir_m is None and ref_dir.name == W2S_SPLIT:
                # Run true paired Direct baseline for gpt-4.1 on the exact same split (zero misattribution!)
                terra_dir_agent = DirectChatBaseline(openai_client=oai_client, model=terra_judge_model)
                terra_dir_eps = await run_when2speak_eval(
                    agent=terra_dir_agent,
                    turns=eval_turns,
                    output_jsonl=RUN_DIR / f"w2s_direct_{terra_judge_model}.jsonl",
                    resume=True,
                    max_concurrency=8,
                )
                terra_dir_m = compute_when2speak_metrics(y_true_w2s, terra_dir_eps)
            w2s_cross_rows.append({
                "family": "OpenAI-4.1",
                "model": terra_judge_model,
                "source_split": d.get("mode", ref_dir.name),
                "direct_macro_f1": terra_dir_m.get("macro_f1") if terra_dir_m else None,
                "direct_fir": terra_dir_m.get("fir") if terra_dir_m else None,
                "direct_mir": terra_dir_m.get("mir") if terra_dir_m else None,
                "dial_macro_f1": ms["DIAL_Skill_terra"]["macro_f1"],
                "dial_speak_f1": ms["DIAL_Skill_terra"]["speak_f1"],
                "dial_fir": ms["DIAL_Skill_terra"]["fir"],
                "dial_mir": ms["DIAL_Skill_terra"]["mir"],
                "dial_auroc": ms["DIAL_Skill_terra"]["auroc"],
                "mcnemar_p": terra_mc_p,
            })
        break

for spec in HEALTHY_MODELS:
    if not spec.get("run_w2s", True):
        continue
    mid = spec["id"]
    slug = mid.replace("/", "_").replace(":", "_")
    cli = _client_for(spec["provider"])
    max_conc = 4 if spec["provider"] == "autodl" else 8
    print(f"\n=== When2Speak [{spec['family']}] {mid} ===")
    try:
        # 1. Per-model validation calibration (<= 200 turns, strictly leak-free)
        calib_agent = DialSkillAgent(
            triage_client=triage_client, openai_client=cli, gate_model=mid, executor_model=mid, cfg=DIAL_CFG, method_name=f"DIAL_{slug}"
        )
        val_eps = await run_when2speak_eval(
            agent=calib_agent, turns=val_turns, output_jsonl=RUN_DIR / f"w2s_val_{slug}.jsonl", resume=True, generate_text=False, max_concurrency=max_conc
        )
        cal = calibrate_when2speak_thresholds(
            val_turns=val_turns,
            val_episodes=val_eps,
            eval_turns=eval_turns,
            lambda_grid=EXP_CFG["when2speak"]["calibration"]["lambda_grid"],
            tau_dm_grid=EXP_CFG["when2speak"]["calibration"]["tau_dm_grid"],
            theta_floor=DIAL_CFG.gate.theta_floor,
            max_fir=0.11,
            max_mir=0.45,
        )
        tuned_cfg = load_dial_skill_config(
            overrides={"gate": {"lambda_disruption": cal["best_lambda"], "tau_dm": cal["best_tau_dm"]}}
        )

        # 2. Evaluate Direct Chat (B0) vs DIAL-Skill on identical eval_turns
        direct_agent = DirectChatBaseline(openai_client=cli, model=mid)
        dial_agent = DialSkillAgent(
            triage_client=triage_client, openai_client=cli, gate_model=mid, executor_model=mid, cfg=tuned_cfg, method_name=f"DIAL_{slug}"
        )
        dir_eps = await run_when2speak_eval(
            agent=direct_agent, turns=eval_turns, output_jsonl=RUN_DIR / f"w2s_direct_{slug}.jsonl", resume=True, max_concurrency=max_conc
        )
        dial_eps = await run_when2speak_eval(
            agent=dial_agent, turns=eval_turns, output_jsonl=RUN_DIR / f"w2s_dial_{slug}.jsonl", resume=True, generate_text=False, max_concurrency=max_conc
        )

        m_dir = compute_when2speak_metrics(y_true_w2s, dir_eps)
        m_dial = compute_when2speak_metrics(y_true_w2s, dial_eps)
        mc = mcnemar_test(y_true_w2s, [e.gate.binary_speak for e in dial_eps], [e.gate.binary_speak for e in dir_eps])

        row = {
            "family": spec["family"],
            "model": mid,
            "provider": spec["provider"],
            "source_split": W2S_SPLIT,
            "calibration": cal,
            "direct_macro_f1": m_dir["macro_f1"],
            "direct_fir": m_dir["fir"],
            "direct_mir": m_dir["mir"],
            "dial_macro_f1": m_dial["macro_f1"],
            "dial_speak_f1": m_dial["speak_f1"],
            "dial_fir": m_dial["fir"],
            "dial_mir": m_dial["mir"],
            "dial_auroc": m_dial["auroc"],
            "dial_token_reduction_pct": m_dial["token_reduction_pct"],
            "mcnemar_chi2": mc["chi2_stat"],
            "mcnemar_p": mc["p_value"],
        }
        w2s_cross_rows.append(row)
        print(
            f"  [{mid:22s}] Direct F1={m_dir['macro_f1']:.3f} (FIR={m_dir['fir']:.3f}) -> "
            f"DIAL F1={m_dial['macro_f1']:.3f} (FIR={m_dial['fir']:.3f}, AUROC={m_dial['auroc']:.3f}) | "
            f"Delta={m_dial['macro_f1'] - m_dir['macro_f1']:+.3f} (p={mc['p_value']:.4g})"
        )
    except Exception as exc:
        print(f"  [ERROR] Skipped W2S for {mid}: {type(exc).__name__}: {exc}")

(RUN_DIR / "cross_model_when2speak.json").write_text(json.dumps(w2s_cross_rows, indent=2), encoding="utf-8")
print("\nSaved cross_model_when2speak.json (", len(w2s_cross_rows), "models)")


## Step 3: Cross-Model PageMantle Generalization + Cross-Family Blinded Judge Agreement

In [ ]:
from dial_skill.benchmarks.pagemantle_env import PageMantleEnv
from dial_skill.eval.judge import BlindedDualJudge, compute_cohens_kappa, compute_spearman_rho
from dial_skill.eval.metrics import compute_pagemantle_metrics
from dial_skill.eval.runners import run_pagemantle_eval
from dial_skill.schemas import PageMantleTask

def _load_pm(name: str):
    p = DRIVE_ROOT / "datasets" / "pagemantle_ext" / f"{name}.jsonl"
    return [PageMantleTask.model_validate_json(line) for line in p.read_text(encoding="utf-8").splitlines() if line.strip()]

pm_official = _load_pm("pm_official_8")
pm_eval_32 = _load_pm("pm_eval_32")
if PM_SCOPE == "full_40":
    pm_tasks = pm_official + pm_eval_32
else:
    pm_sub = [t for t in pm_eval_32 if t.expected_action == "INTERVENE"][:10] + [
        t for t in pm_eval_32 if t.expected_action == "SILENT"
    ][:6]
    pm_tasks = pm_official + pm_sub

env = PageMantleEnv(dataset_dir=DRIVE_ROOT / "datasets" / "pagemantle")
env.ensure_extracted()
primary_judge = BlindedDualJudge(openai_client=oai_client, primary_judge_model=terra_judge_model)
print(f"PageMantle Cross-Model suite: {len(pm_tasks)} tasks (traps={sum(1 for t in pm_tasks if t.expected_action=='SILENT')})")

pm_cross_rows = []

# Read-only reference from Notebook 04 full run (gpt-4.1-mini & gpt-4.1)
ref_pm_pre = DRIVE_ROOT / "runs" / "main_pagemantle" / "full" / "pre_evolution_metrics.json"
if ref_pm_pre.is_file():
    pm_ref = json.loads(ref_pm_pre.read_text(encoding="utf-8"))
    if "B0_direct_luna" in pm_ref and "DIAL_Skill_v0_luna" in pm_ref:
        pm_cross_rows.append({
            "family": "OpenAI-4.1",
            "model": luna_ref_model,
            "n_tasks": pm_ref["DIAL_Skill_v0_luna"]["n_tasks"],
            "direct_overall": pm_ref["B0_direct_luna"]["overall_score"],
            "dial_overall": pm_ref["DIAL_Skill_v0_luna"]["overall_score"],
            "dial_silence_accuracy": pm_ref["DIAL_Skill_v0_luna"]["silence_accuracy"],
            "dial_action_recall": pm_ref["DIAL_Skill_v0_luna"]["action_recall"],
            "dial_audience": pm_ref["DIAL_Skill_v0_luna"]["audience_and_location"],
            "dial_cost_per_task_usd": pm_ref["DIAL_Skill_v0_luna"]["cost_per_task_usd"],
        })
    if "B0_direct_terra" in pm_ref and "DIAL_Skill_v0_terra" in pm_ref:
        pm_cross_rows.append({
            "family": "OpenAI-4.1",
            "model": terra_judge_model,
            "n_tasks": pm_ref["DIAL_Skill_v0_terra"]["n_tasks"],
            "direct_overall": pm_ref["B0_direct_terra"]["overall_score"],
            "dial_overall": pm_ref["DIAL_Skill_v0_terra"]["overall_score"],
            "dial_silence_accuracy": pm_ref["DIAL_Skill_v0_terra"]["silence_accuracy"],
            "dial_action_recall": pm_ref["DIAL_Skill_v0_terra"]["action_recall"],
            "dial_audience": pm_ref["DIAL_Skill_v0_terra"]["audience_and_location"],
            "dial_cost_per_task_usd": pm_ref["DIAL_Skill_v0_terra"]["cost_per_task_usd"],
        })

sample_episodes_for_cross_judge = []

for spec in HEALTHY_MODELS:
    if not spec.get("run_pm", True):
        continue
    mid = spec["id"]
    slug = mid.replace("/", "_").replace(":", "_")
    cli = _client_for(spec["provider"])
    print(f"\n=== PageMantle [{spec['family']}] {mid} ===")
    try:
        dir_agent = DirectChatBaseline(openai_client=cli, model=mid)
        dial_agent = DialSkillAgent(
            triage_client=triage_client,
            openai_client=cli,
            gate_model=mid,
            executor_model=mid,
            skill_store=SkillStore.from_json_file(),
            cfg=DIAL_CFG,
            method_name=f"DIAL_{slug}",
        )
        dir_eps = await run_pagemantle_eval(
            agent=dir_agent, tasks=pm_tasks, env=env, output_jsonl=RUN_DIR / f"pm_direct_{slug}.jsonl", resume=True
        )
        dial_eps = await run_pagemantle_eval(
            agent=dial_agent, tasks=pm_tasks, env=env, output_jsonl=RUN_DIR / f"pm_dial_{slug}.jsonl", resume=True
        )
        dir_scores = [await primary_judge.grade_pagemantle_episode(t, e) for t, e in zip(pm_tasks, dir_eps)]
        dial_scores = [await primary_judge.grade_pagemantle_episode(t, e) for t, e in zip(pm_tasks, dial_eps)]
        (RUN_DIR / f"pm_judge_dial_{slug}.jsonl").write_text("\n".join(s.model_dump_json() for s in dial_scores) + "\n", encoding="utf-8")

        m_dir = compute_pagemantle_metrics(pm_tasks, dir_eps, dir_scores)
        m_dial = compute_pagemantle_metrics(pm_tasks, dial_eps, dial_scores)

        pm_cross_rows.append({
            "family": spec["family"],
            "model": mid,
            "provider": spec["provider"],
            "n_tasks": len(pm_tasks),
            "direct_overall": m_dir["overall_score"],
            "direct_silence_accuracy": m_dir["silence_accuracy"],
            "dial_overall": m_dial["overall_score"],
            "dial_silence_accuracy": m_dial["silence_accuracy"],
            "dial_action_recall": m_dial["action_recall"],
            "dial_audience": m_dial["audience_and_location"],
            "dial_timeout_rate": m_dial["timeout_rate"],
            "dial_cost_per_task_usd": m_dial["cost_per_task_usd"],
        })
        print(
            f"  [{mid:22s}] Direct={m_dir['overall_score']:.2f} (SilAcc={m_dir['silence_accuracy']:.2f}) -> "
            f"DIAL={m_dial['overall_score']:.2f} (SilAcc={m_dial['silence_accuracy']:.2f}, ActRec={m_dial['action_recall']:.2f}, Aud={m_dial['audience_and_location']:.2f}) | "
            f"Delta={m_dial['overall_score'] - m_dir['overall_score']:+.2f}"
        )
        # Collect a stratified subset (DIAL action + DIAL silence trap + Direct action/trap) for meaningful rho & kappa
        dial_intervene = [(t, e, s) for t, e, s in zip(pm_tasks, dial_eps, dial_scores) if t.expected_action == "INTERVENE"][:2]
        dial_silent = [(t, e, s) for t, e, s in zip(pm_tasks, dial_eps, dial_scores) if t.expected_action == "SILENT"][:2]
        dir_intervene = [(t, e, s) for t, e, s in zip(pm_tasks, dir_eps, dir_scores) if t.expected_action == "INTERVENE"][:1]
        dir_silent = [(t, e, s) for t, e, s in zip(pm_tasks, dir_eps, dir_scores) if t.expected_action == "SILENT"][:1]
        sample_episodes_for_cross_judge.extend(dial_intervene + dial_silent + dir_intervene + dir_silent)
    except Exception as exc:
        print(f"  [ERROR] Skipped PageMantle for {mid}: {type(exc).__name__}: {exc}")

(RUN_DIR / "cross_model_pagemantle.json").write_text(json.dumps(pm_cross_rows, indent=2), encoding="utf-8")

# --- Cross-Family Blinded Judge Agreement (OpenAI gpt-4.1 vs AutoDL non-OpenAI judge) ---
autodl_healthy = [m["id"] for m in HEALTHY_MODELS if m["provider"] == "autodl"]
cross_judge_report = {}
if autodl_healthy and sample_episodes_for_cross_judge:
    # Prefer largest reasoning/flagship AutoDL model as independent cross-family judge
    alt_judge_model = next((m for m in ("Qwen3.5-397B-A17B", "DeepSeek-V4.1-Flash", "GLM-5.3-flash") if m in autodl_healthy), autodl_healthy[0])
    alt_judge = BlindedDualJudge(openai_client=autodl_client, primary_judge_model=alt_judge_model)
    subset = sample_episodes_for_cross_judge[:30]
    p_scores, s_scores = [], []
    for t, e, prim_s in subset:
        try:
            sec_s = await alt_judge.grade_pagemantle_episode(t, e)
            p_scores.append(prim_s.overall_score)
            s_scores.append(sec_s.overall_score)
        except Exception as exc:
            print(f"  [Cross-Judge skip {t.task_id}]: {exc}")
    if len(p_scores) >= 4:
        rho_res = compute_spearman_rho(p_scores, s_scores)
        kappa = compute_cohens_kappa([int(x >= 6.0) for x in p_scores], [int(x >= 6.0) for x in s_scores])
        mae = float(sum(abs(a - b) for a, b in zip(p_scores, s_scores)) / len(p_scores))
        cross_judge_report = {
            "primary_judge": terra_judge_model,
            "secondary_judge": alt_judge_model,
            "n_episodes": len(p_scores),
            "spearman_rho": rho_res["rho"],
            "spearman_p": rho_res["p_value"],
            "cohens_kappa": kappa,
            "mae": round(mae, 4),
            "primary_scores": p_scores,
            "secondary_scores": s_scores,
        }
        (RUN_DIR / "cross_family_judge_agreement.json").write_text(json.dumps(cross_judge_report, indent=2), encoding="utf-8")
        print(f"\nCross-Family Judge Agreement ({terra_judge_model} vs {alt_judge_model}, n={len(p_scores)}): rho={rho_res['rho']:.3f}, kappa={kappa:.3f}, MAE={mae:.2f}")


## Step 4: Generate Incremental LaTeX Tables & Vector PDF Figures (New Filenames Only)

In [ ]:
from dial_skill.eval.report import (
    generate_tab_cross_model_pagemantle,
    generate_tab_cross_model_when2speak,
    plot_cross_family_judge_agreement,
    plot_cross_model_pagemantle_gain,
    plot_cross_model_w2s_gain,
)

TABLES_DIR = DRIVE_ROOT / "paper_artifacts" / "tables"
FIGS_DIR = DRIVE_ROOT / "paper_artifacts" / "figures"

if w2s_cross_rows:
    print("Wrote:", generate_tab_cross_model_when2speak(w2s_cross_rows, TABLES_DIR / "tab_cross_model_when2speak.tex"))
    print("Wrote:", plot_cross_model_w2s_gain(w2s_cross_rows, FIGS_DIR / "fig_cross_model_w2s_gain.pdf"))

if pm_cross_rows:
    print("Wrote:", generate_tab_cross_model_pagemantle(pm_cross_rows, TABLES_DIR / "tab_cross_model_pagemantle.tex"))
    print("Wrote:", plot_cross_model_pagemantle_gain(pm_cross_rows, FIGS_DIR / "fig_cross_model_pagemantle_gain.pdf"))

if cross_judge_report:
    print(
        "Wrote:",
        plot_cross_family_judge_agreement(
            primary_scores=cross_judge_report["primary_scores"],
            secondary_scores=cross_judge_report["secondary_scores"],
            primary_label=cross_judge_report["primary_judge"],
            secondary_label=cross_judge_report["secondary_judge"],
            spearman_rho=cross_judge_report["spearman_rho"],
            out_path=FIGS_DIR / "fig_cross_family_judge_agreement.pdf",
        ),
    )

print(f"\nSession spend -> OpenAI=${oai_client.usage.total_cost_usd:.4f} | AutoDL(est)=${autodl_client.usage.total_cost_usd if autodl_client else 0.0:.4f} | Jev=${triage_client.total_cost_usd:.4f}")
